# Resident Health Trajectory Predictor
**Lighthouse Sanctuary — ML Pipeline (Lighter)**

Predict next-month general health score for a resident using lag features and compliance indicators.

---
## 1. Problem Framing

### Business Problem
Early detection of declining health trends allows social workers to proactively schedule medical checkups, nutritional interventions, or counseling before a resident's health score drops significantly. A simple trajectory model provides an early-warning signal.

### ML Task
- **Type:** Regression  
- **Target:** `general_health_score` at time T (score 1–5)  
- **Key features:** prior two months' scores, BMI trend, checkup compliance, incident count  
- **Unit:** One record = one resident × one month  

### Models
- Linear Regression (baseline, interpretable)
- Gradient Boosting Regressor (light, no deep tuning)

### Deployment
No API endpoint. Outputs a per-resident health forecast table for social worker dashboards.

---
## 2. Data Acquisition, Preparation & Exploration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import subprocess, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer

NB_DIR    = Path().resolve()
ML_DIR    = NB_DIR.parent if NB_DIR.name == 'notebooks' else NB_DIR / 'ml-pipelines'
RAW_DIR   = ML_DIR.parent / 'MLPipelines' / 'Data' / 'lighthouse_v7'

# Load raw health records (need time-series structure)
hlth = pd.read_csv(RAW_DIR / 'health_wellbeing_records.csv', low_memory=False)
inc  = pd.read_csv(RAW_DIR / 'incident_reports.csv', low_memory=False)

hlth['record_date'] = pd.to_datetime(hlth['record_date'], errors='coerce')
inc['incident_date'] = pd.to_datetime(inc['incident_date'], errors='coerce')

print(f'Health records: {hlth.shape}')
print(f'Incident reports: {inc.shape}')
hlth.head()

In [ ]:
# Health score distribution and trends
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(hlth['general_health_score'].dropna(), bins=30, color='#2ecc71', edgecolor='white')
axes[0].set_title('Distribution of Health Scores')
axes[0].set_xlabel('Score (1–5)')

# Monthly average health score over time
hlth['month'] = hlth['record_date'].dt.to_period('M')
monthly_avg = hlth.groupby('month')['general_health_score'].mean()
monthly_avg.plot(ax=axes[1], color='#2ecc71', marker='o', ms=4)
axes[1].set_title('Monthly Avg Health Score Over Time')
axes[1].set_ylabel('Avg Score')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Build lag features: for each resident, sort by date and create t-1, t-2 features
hlth_sorted = hlth.sort_values(['resident_id', 'record_date'])

hlth_lag = hlth_sorted.copy()
hlth_lag['health_lag1'] = hlth_lag.groupby('resident_id')['general_health_score'].shift(1)
hlth_lag['health_lag2'] = hlth_lag.groupby('resident_id')['general_health_score'].shift(2)
hlth_lag['bmi_lag1']    = hlth_lag.groupby('resident_id')['bmi'].shift(1)
hlth_lag['bmi_change']  = hlth_lag['bmi'] - hlth_lag['bmi_lag1']
hlth_lag['checkup_done'] = hlth_lag['medical_checkup_done'].astype(str).str.lower().map({'true':1,'false':0}).fillna(0)

# Add incident count per resident in prior 30 days
# Simplified: use total incident count as a static feature joined per resident
inc_count = inc.groupby('resident_id').size().reset_index(name='total_incidents')
hlth_lag = hlth_lag.merge(inc_count, on='resident_id', how='left')
hlth_lag['total_incidents'] = hlth_lag['total_incidents'].fillna(0)

# Drop rows where we can't compute lag features
model_df = hlth_lag.dropna(subset=['health_lag1', 'health_lag2', 'general_health_score'])
print(f'Lag dataset: {model_df.shape}')
model_df[['general_health_score','health_lag1','health_lag2','bmi','checkup_done']].describe()

In [ ]:
# Correlation of lag features with current score
corr_cols = ['general_health_score','health_lag1','health_lag2','bmi_change','checkup_done','total_incidents']
existing = [c for c in corr_cols if c in model_df.columns]
corr = model_df[existing].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0, square=True)
plt.title('Feature Correlations — Health Trajectory')
plt.tight_layout()
plt.show()

---
## 3. Modeling & Feature Selection

In [ ]:
FEATURES = ['health_lag1', 'health_lag2', 'bmi_change', 'checkup_done', 'total_incidents']
FEATURES = [f for f in FEATURES if f in model_df.columns]
TARGET = 'general_health_score'

X = model_df[FEATURES].fillna(model_df[FEATURES].median())
y = model_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

# Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

lr_mae = mean_absolute_error(y_test, y_pred_lr)
lr_r2  = r2_score(y_test, y_pred_lr)

# Gradient Boosting (light)
gb = GradientBoostingRegressor(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)

gb_mae = mean_absolute_error(y_test, y_pred_gb)
gb_r2  = r2_score(y_test, y_pred_gb)

print(f'\nLinear Regression:   MAE={lr_mae:.4f}, R²={lr_r2:.4f}')
print(f'Gradient Boosting:   MAE={gb_mae:.4f}, R²={gb_r2:.4f}')

---
## 4. Evaluation & Interpretation

In [ ]:
# Predicted vs Actual
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, y_pred, label, color in [
    (axes[0], y_pred_lr, 'Linear Regression', '#3498db'),
    (axes[1], y_pred_gb, 'Gradient Boosting', '#2ecc71')
]:
    ax.scatter(y_test, y_pred, alpha=0.4, s=15, color=color)
    lim = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax.plot(lim, lim, 'r--', lw=1)
    ax.set_title(f'{label}\nMAE={mean_absolute_error(y_test, y_pred):.4f}, R²={r2_score(y_test, y_pred):.4f}')
    ax.set_xlabel('Actual Health Score')
    ax.set_ylabel('Predicted')

plt.suptitle('Predicted vs Actual Health Score', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Linear regression coefficients
coef_df = pd.DataFrame({'Feature': FEATURES, 'Coefficient': lr.coef_}).sort_values('Coefficient')
colors = ['#2ecc71' if c > 0 else '#e74c3c' for c in coef_df['Coefficient']]

plt.figure(figsize=(8, 4))
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Linear Regression Coefficients\n(positive = increases predicted health score)')
plt.tight_layout()
plt.show()

print('Coefficients:')
for _, row in coef_df.sort_values('Coefficient', ascending=False).iterrows():
    print(f'  {row["Feature"]:25s}: {row["Coefficient"]:+.4f}')

In [ ]:
# Per-resident forecast: flag residents with declining trajectory
latest = hlth_sorted.groupby('resident_id').last().reset_index()
latest['health_lag1'] = hlth_sorted.groupby('resident_id')['general_health_score'].shift(1).groupby(hlth_sorted['resident_id']).last().values

# Simplified: use the last two scores to predict next
all_lag = hlth_sorted.groupby('resident_id').agg(
    last_score=('general_health_score', 'last'),
    second_last=('general_health_score', lambda x: x.iloc[-2] if len(x) >= 2 else x.iloc[-1]),
    last_bmi=('bmi', 'last'),
    checkup_compliance=('medical_checkup_done', lambda x: x.astype(str).str.lower().eq('true').mean())
).reset_index()

all_lag = all_lag.merge(inc_count, on='resident_id', how='left')
all_lag['total_incidents'] = all_lag['total_incidents'].fillna(0)
all_lag['bmi_change'] = 0  # no prior period available, set to 0

forecast_features = ['last_score', 'second_last', 'bmi_change', 'checkup_compliance', 'total_incidents']
forecast_X = all_lag[forecast_features].fillna(all_lag[forecast_features].median())
forecast_X.columns = FEATURES

all_lag['predicted_next_health'] = gb.predict(forecast_X)
all_lag['health_trend'] = all_lag['predicted_next_health'] - all_lag['last_score']
all_lag['alert'] = all_lag['health_trend'] < -0.3

print('=== Residents with Declining Health Trajectory ===')
alerts = all_lag[all_lag['alert']].sort_values('health_trend')
print(f'{len(alerts)} residents flagged for declining health')
print(alerts[['resident_id','last_score','predicted_next_health','health_trend']].head(10).to_string(index=False))

---
## 5. Causal and Relationship Analysis

In [ ]:
# Does checkup compliance improve health trajectory?
print('=== Health Score by Checkup Compliance ===')
compliance_group = model_df.copy()
compliance_group['high_compliance'] = compliance_group['checkup_done'] >= 0.5
analysis = compliance_group.groupby('high_compliance')['general_health_score'].agg(['mean','std','count'])
analysis.index = ['Low Compliance', 'High Compliance']
print(analysis.to_string())

# Does incident count predict worse health?
print('\n=== Health Score vs Incident Count ===')
inc_bins = pd.cut(model_df['total_incidents'], bins=[-1,0,2,5,100], labels=['0','1-2','3-5','6+'])
inc_health = model_df.groupby(inc_bins, observed=True)['general_health_score'].mean()
print(inc_health.to_string())

plt.figure(figsize=(7, 4))
inc_health.plot(kind='bar', color='#e74c3c')
plt.title('Avg Health Score by Incident Count')
plt.ylabel('Avg Health Score')
plt.xlabel('Total Incidents')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---
## 6. Deployment Notes

No API endpoint is deployed for this model.

### Integration Path
- The per-resident forecast table (cell above) can be exported as a CSV or surfaced in the admin dashboard as a "Health Alert" panel
- Trigger an automatic notification to the assigned social worker when `health_trend < -0.3`

### If promoting to production
- Save the GB model: `joblib.dump(gb, MODEL_DIR / 'health_trajectory_model.pkl')`
- Add endpoint `POST /predict/health-trajectory` accepting `{resident_id, lag_features}`
- Re-run monthly after health records are updated

### Limitations
- Lag features require at least 2 prior records per resident
- New residents (<2 months) cannot be scored — use average safehouse health score as prior
- Monthly aggregation loses within-month variation